In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_del = df[
    df["ISSUE"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.startswith("delete")
]

df_del

,ISSUE,Network ID,Network Name,Native ID,Station ID,Unique Names,Unique Location (String),Unique Records,Uniq Obs Freqs,Variables,...,SITE TYPE,OWNER,FIRE CENTRE,FIRE ZONE,LATITUDE,LONGITUDE,ELEVATION,Unnamed: 50,Unnamed: 51,Unnamed: 52
391,Delete,10.0,BC AGRIWeather -> MVRD,T12,1527.0,Chilliwack,1930,BC,daily,36169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
392,Delete,10.0,BC AGRIWeather -> MVRD,T13,1528.0,North Delta,1931,BC,daily,33555,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
393,Delete,10.0,BC AGRIWeather -> MVRD,T15,1529.0,Surrey East,1932,BC,daily,33532,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
394,Delete,10.0,BC AGRIWeather -> MVRD,T17,1530.0,Richmond South,1933,BC,daily,33800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395,Delete,10.0,BC AGRIWeather -> MVRD,T18,1531.0,Burnaby South,1934,BC,daily,36288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
735,Delete - covered elsewhere,9.0,BC-Air,Kamloops Brocklehurst,12940.0,NaN,"0 W, null N, Elev. null m",unknown to unknown,Unspecified,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
736,Delete - covered elsewhere,9.0,BC-Air,Osoyoos Canada Customs,12942.0,NaN,"0 W, null N, Elev. null m",unknown to unknown,Unspecified,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
737,Delete - Defer to record 12170,9.0,BC-Air,Kelowna College -> 102,12941.0,-> Kelowna College,"0 W, null N, Elev. null m",unknown to unknown,Unspecified,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
738,Delete - Site covered elsewhere,9.0,BC-Air,Smithers St Josephs -> 171,12944.0,-> Smithers St Joseph,"0 W, null N, Elev. null m",unknown to unknown,Unspecified,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
station_ids = (
    df_del["Station ID"]
    .dropna()
    .astype(int) 
    .unique()
    .tolist()
)
station_ids[0]


1527

In [6]:
# DELETE FROM obs_raw o
# USING meta_history h
# WHERE o.history_id = h.history_id
#   AND h.station_id = 1527

In [8]:
from tqdm import tqdm
import sqlalchemy as sa

# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:   0%|          | 1/346 [00:06<38:19,  6.67s/it]

Station 1527: flags=0, obs_raw=12759, obs_derived=36, meta_history=1, meta_station=1


Deleting stations:   1%|          | 2/346 [00:15<44:54,  7.83s/it]

Station 1528: flags=0, obs_raw=20186, obs_derived=36, meta_history=1, meta_station=1


Deleting stations:   1%|          | 3/346 [00:23<46:04,  8.06s/it]

...

Station 2519: flags=0, obs_raw=436326, obs_derived=26, meta_history=1, meta_station=1


Deleting stations:  98%|█████████▊| 340/346 [41:11<07:37, 76.31s/it]

Station 13123: flags=0, obs_raw=60284, obs_derived=0, meta_history=1, meta_station=1


Deleting stations: 100%|██████████| 346/346 [41:15<00:00,  7.15s/it]

Station 2491: flags=0, obs_raw=8959, obs_derived=36, meta_history=1, meta_station=1
Station 12940: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 12942: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 12941: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 12944: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
Station 12943: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


### true progress and safe restart, do this:

In [ ]:
for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):
    try:
        with engine.begin() as conn:
            conn.execute(delete_obs_flags, {"station_id": sid})
            conn.execute(delete_obs, {"station_id": sid})
            conn.execute(delete_obs_derived, {"station_id": sid})
            conn.execute(delete_history, {"station_id": sid})
            conn.execute(delete_station, {"station_id": sid})

    except Exception as e:
        tqdm.write(f"❌ Station {sid} failed: {e}")
        continue

### Delete specific station
(Consider Deleting - data is not worth keeping)


In [9]:
from tqdm import tqdm
import sqlalchemy as sa

# List of station_ids to delete
station_ids_to_delete = [12280]  # or a subset



with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations: 100%|██████████| 1/1 [00:58<00:00, 58.15s/it]

Station 12280: flags=0, obs_raw=101546, obs_derived=0, meta_history=1, meta_station=1


In [10]:
check_sql = sa.text("""
SELECT s.station_id, h.station_name, count(o.obs_raw_id) AS n_obs
FROM meta_station s
LEFT JOIN meta_history h ON s.station_id = h.station_id
LEFT JOIN obs_raw o ON h.history_id = o.history_id
WHERE s.station_id = ANY(:station_ids)
GROUP BY s.station_id, h.station_name
""")

with engine.connect() as conn:
    df_check = pd.read_sql(check_sql, conn, params={"station_ids": station_ids})

df_check

,station_id,station_name,n_obs
